# Individual and Market Demand: Perfect Substitutes

Imagine we had two Ride Share companies (X and Y). The six groups have **different perfect-substitutes utility functions** and incomes:

$$ U_i(X,Y)=a_i \text{X}+b_i \text{Y},\qquad p_x x+p_y y\le m_i.$$

We focus on the X ride share service. We vary the **price of X** on the vertical axis and plot **quantity of X** on the horizontal axis. 

In the case of perfect substitutes:

$$\text{X}_i^*(p_X)=\begin{cases}m_i/p_X&\text{if }a_i/p_X>b_i/p_Y,\\0&\text{if }a_i/p_X<b_i/p_Y.\end{cases}$$

Recall that Equal Marginal principle reflects $|MRS| = \text{Price ratio}$:

$$\frac{a_i}{p_X}>\frac{b_i}{p_Y} \implies  \frac{a_i}{b_i}>\frac{P_X}{p_Y} \implies |MRS| > \text{price ratio}$$


Market demand is the **horizontal sum**: $\text{Market for X ride share}(p_X)=\sum_{i=1}^6 \text{X}_i^*(p_X)$.

In [3]:
# Install once if needed (uncomment):
# %pip install ipywidgets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# In classic Jupyter notebook, the default inline backend works well.
# For live Matplotlib figure manipulation, optionally install ipympl and use %matplotlib widget.

In /opt/anaconda3/lib/python3.8/site-packages/matplotlib/mpl-data/stylelib/_classic_test.mplstyle: 
The text.latex.preview rcparam was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In /opt/anaconda3/lib/python3.8/site-packages/matplotlib/mpl-data/stylelib/_classic_test.mplstyle: 
The mathtext.fallback_to_cm rcparam was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In /opt/anaconda3/lib/python3.8/site-packages/matplotlib/mpl-data/stylelib/_classic_test.mplstyle: Support for setting the 'mathtext.fallback_to_cm' rcParam is deprecated since 3.3 and will be removed two minor releases later; use 'mathtext.fallback : 'cm' instead.
In /opt/anaconda3/lib/python3.8/site-packages/matplotlib/mpl-data/stylelib/_classic_test.mplstyle: 
The validate_bool_maybe_none function was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In /opt/anaconda3/lib/python3.8/site-packages/matplotlib/mpl-data/stylelib/_classic_test.mpl

In [4]:
# EDIT THESE SIX CONSUMERS to fit your lecture.
# Utility is U_i(X,Y) = a_i * Ubers + b_i * Lyfts.
consumers = [
    {"name": "Group 1", "a": 2.0, "b": 5.0, "income": 34},
    {"name": "Group 2", "a": 2.5, "b": 4.0, "income": 66},
    {"name": "Group 3", "a": 14.0, "b": 6.0, "income": 22},
    {"name": "Group 4", "a": 10.0, "b": 5.0, "income": 66},
    {"name": "Group 5", "a": 8.0, "b": 6.0, "income": 49},
    {"name": "Group 6", "a": 6.0, "b": 4.0, "income": 37},
]

# Price range of X to plot (strictly positive), and initial price of Y.
PX_MIN, PX_MAX = 0.25, 6.0
INITIAL_PY = 1.0
TIE_RULE = "x"  # "x" or "y"; ties admit multiple optimal bundles.

In [5]:
def x_demand(px, py, income, a, b, tie_rule="x"):
    """Demand for X for utility a*X+b*Y and budget px*X+py*Y<=income.

    px may be a scalar or NumPy array. Positive px,py,a,b required.
    """
    px = np.asarray(px, dtype=float)
    if py <= 0 or np.any(px <= 0) or a <= 0 or b <= 0:
        raise ValueError("Prices and utility weights must be positive.")

    threshold = a * py / b
    buys_x = (
        (px < threshold)
        | (
            np.isclose(px, threshold, atol=1e-10, rtol=0)
            & (tie_rule == "x")
        )
    )

    return np.where(buys_x, income / px, 0.0)


def market_demand(px, py, incomes):
    return sum(
        x_demand(px, py, m, c["a"], c["b"], TIE_RULE)
        for c, m in zip(consumers, incomes)
    )


def draw_demand(py, px_current, **income_values):

    incomes = [income_values[f"income_{i}"] for i in range(6)]
    px_grid = np.linspace(PX_MIN, PX_MAX, 2000)

    # ---------------------------------------------------
    # Common x-axis for the SIX individual consumers
    # ---------------------------------------------------
    max_individual_q = max(incomes) / PX_MIN
    individual_xlim = (0, max_individual_q * 1.05)

    # Market demand has its OWN x-axis
    max_market_q = sum(incomes) / PX_MIN
    market_xlim = (0, max_market_q * 1.05)

    # ---------------------------------------------------
    # Create figure and subplots
    # ---------------------------------------------------
    fig = plt.figure(figsize=(14, 11), constrained_layout=True)
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1.35])

    axes = [
        fig.add_subplot(gs[r, c])
        for r in range(2)
        for c in range(3)
    ]

    ax_market = fig.add_subplot(gs[2, :])

    # ---------------------------------------------------
    # Individual demand curves
    # ---------------------------------------------------
    for i, (ax, c, m) in enumerate(zip(axes, consumers, incomes)):

        threshold = c["a"] * py / c["b"]

        active = px_grid < threshold
        inactive = px_grid > threshold

        # Consumer purchases Ubers
        ax.plot(
            m / px_grid[active],
            px_grid[active],
            lw=2,
            color=f"C{i}"
        )

        # Consumer purchases no Ubers
        if inactive.any():
            ax.plot(
                np.zeros(inactive.sum()),
                px_grid[inactive],
                lw=2,
                color=f"C{i}"
            )

        # Current price and quantity demanded
        q_now = float(
            x_demand(
                px_current, py, m,
                c["a"], c["b"], TIE_RULE
            )
        )

        ax.scatter(
            [q_now], [px_current],
            color="black",
            zorder=4,
            s=30
        )

        ax.axhline(
            px_current,
            ls=":",
            color="gray",
            lw=0.9
        )

        ax.set_title(
            f"{c['name']}: U={c['a']:g}X+{c['b']:g}Y, m=${m:g}",
            fontsize=10
        )

        ax.set_xlabel("Quantity of X")
        ax.set_ylabel("Price of X")

        # Shared limits for the six individuals
        ax.set_ylim(PX_MIN, PX_MAX)
        ax.set_xlim(individual_xlim)

        ax.grid(alpha=0.2)

    # ---------------------------------------------------
    # Market demand curve
    # ---------------------------------------------------
    breaks = sorted(
        {PX_MIN, PX_MAX}
        | {
            c["a"] * py / c["b"]
            for c in consumers
            if PX_MIN < c["a"] * py / c["b"] < PX_MAX
        }
    )

    for low, high in zip(breaks[:-1], breaks[1:]):

        segment_px = np.linspace(low, high, 300)[1:-1]

        segment_q = market_demand(
            segment_px, py, incomes
        )

        ax_market.plot(
            segment_q,
            segment_px,
            color="tab:blue",
            lw=2.5
        )

    # Current market quantity demanded
    q_market_now = float(
        market_demand(px_current, py, incomes)
    )

    ax_market.scatter(
        [q_market_now],
        [px_current],
        color="black",
        zorder=5,
        s=60,
        label=(
            f"At P X=${px_current:.2f}: "
            f"Market Q of X={q_market_now:.1f}"
        )
    )

    ax_market.axhline(
        px_current,
        color="gray",
        linestyle=":",
        lw=1
    )

    ax_market.set_title(
        "MARKET DEMAND = horizontal sum of six individual demands",
        fontsize=13
    )

    ax_market.set_xlabel("Total quantity of X")
    ax_market.set_ylabel("Price of X")

    # Independent x-axis for market demand
    ax_market.set_ylim(PX_MIN, PX_MAX)
    ax_market.set_xlim(market_xlim)

    ax_market.grid(alpha=0.2)
    ax_market.legend(loc="upper right")

    plt.show()

    # ---------------------------------------------------
    # Print demand quantities
    # ---------------------------------------------------
    print(
        f"At P of X=${px_current:.2f} "
        f"and P of Y=${py:.2f}:"
    )

    individual_demands = [
        round(
            float(
                x_demand(
                    px_current, py, m,
                    c["a"], c["b"], TIE_RULE
                )
            ),
            2
        )
        for c, m in zip(consumers, incomes)
    ]

    print("Individual X demand:", individual_demands)
    print(f"Market X demand: {q_market_now:.2f}")

In [6]:
# Interactive controls: drag prices or change consumers' incomes.
style = {"description_width": "150px"}
price_y_slider = widgets.FloatSlider(value=INITIAL_PY, min=0.5, max=3.0, step=0.1,
                                     description="Price of Y", style=style,
                                     continuous_update=False)
price_x_slider = widgets.FloatSlider(value=1.5, min=PX_MIN, max=PX_MAX, step=0.05,
                                     description="Current Price of X", style=style,
                                     continuous_update=False)
income_sliders = {
    f"income_{i}": widgets.IntSlider(value=c["income"], min=10, max=200, step=10,
                                      description=f"Group {i + 1} income", style=style,
                                      continuous_update=False)
    for i, c in enumerate(consumers)
}

controls = {"py": price_y_slider, "px_current": price_x_slider, **income_sliders}
output = widgets.interactive_output(draw_demand, controls)
display(widgets.VBox([
    widgets.HTML("<b>Prices</b>"), widgets.HBox([price_y_slider, price_x_slider]),
    widgets.HTML("<b>Consumer incomes</b>"),
    widgets.GridBox(list(income_sliders.values()),
                    layout=widgets.Layout(grid_template_columns="repeat(2, minmax(300px, 1fr))")),
    output,
]))